In [ ]:
# ================================================================
# Direct file read from Colab
# ================================================================

# If needed, uncomment:
# !pip install -q pandas matplotlib seaborn plotly

import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------
# FILE PATH IN COLAB
# ---------------------------------------------------------------
file_path = 'Table1.csv'

def load_csv() -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    return df


# ---------------------------------------------------------------
# FIGURE 1 — HIV prevalence among patients with vs. without TB
# ---------------------------------------------------------------
def figure_1_hiv_tb_coinfection(df: pd.DataFrame) -> None:
    text_cols = [c for c in df.columns if df[c].dtype == object]
    df_text = df[text_cols].fillna('')

    cond_tb_text = df_text.apply(
        lambda x: x.astype(str).str.contains(r'\b(tuberculose|tuberculosis|tb)\b', case=False, regex=True)
    ).any(axis=1)
    cond_hiv_text = df_text.apply(
        lambda x: x.astype(str).str.contains(
            r'\b(hiv|aids|sida|vírus da imunodeficiência|human immunodeficiency virus)\b',
            case=False,
            regex=True,
        )
    ).any(axis=1)

    cols_infectious = [
        c for c in df.columns if ('infeccios' in c.lower() or 'ciap' in c.lower() or 'choice=' in c.lower())
    ]

    cond_tb_struct = pd.Series(False, index=df.index)
    cond_hiv_struct = pd.Series(False, index=df.index)

    for c in cols_infectious:
        is_checked = df[c].astype(str).str.lower().str.strip().isin(
            ['checked', '1', '1.0', 'sim', 'yes', 'verdadeiro', 'true']
        )

        if 'tuberculose' in c.lower() or 'tuberculosis' in c.lower() or 'a70' in c.lower():
            cond_tb_struct = cond_tb_struct | is_checked
        if 'hiv' in c.lower() or 'aids' in c.lower() or 'b90' in c.lower():
            cond_hiv_struct = cond_hiv_struct | is_checked

    df = df.copy()
    df['Has_TB'] = (cond_tb_text | cond_tb_struct).astype(int)
    df['Has_HIV'] = (cond_hiv_text | cond_hiv_struct).astype(int)

    df_patients = df.groupby('Record ID')[['Has_TB', 'Has_HIV']].max().reset_index()
    df_patients['Group'] = df_patients['Has_TB'].map({0: 'Without Tuberculosis', 1: 'With Tuberculosis'})
    df_patients['HIV_Status'] = df_patients['Has_HIV'].map({0: 'Negative', 1: 'Positive'})

    raw_table_name = 'TeleSAP_TB_HIV_Raw_Table_EN.csv'
    df_patients[['Record ID', 'Group', 'HIV_Status']].to_csv(raw_table_name, index=False, encoding='utf-8-sig')
    print(f"Success! Patient-level table saved as '{raw_table_name}'.\n")

    df_agg = df_patients.groupby('Group')['Has_HIV'].mean().mul(100).reset_index(name='HIV_Prevalence')
    order = ['Without Tuberculosis', 'With Tuberculosis']
    df_agg['Group'] = pd.Categorical(df_agg['Group'], categories=order, ordered=True)
    df_agg = df_agg.sort_values('Group')

    sns.set_theme(style='whitegrid')
    plt.figure(figsize=(10, 6))
    colors = ['#4a8ec2', '#d05c53']

    ax = sns.barplot(data=df_agg, x='Group', y='HIV_Prevalence', palette=colors)
    plt.title(
        'Co-infection Prevalence: HIV/AIDS Among Patients With Tuberculosis\n'
        '(Data Extracted from the TeleSAP Database)',
        fontsize=15,
        pad=20,
    )
    plt.xlabel('Group', fontsize=12)
    plt.ylabel('HIV Prevalence (%)', fontsize=12)
    plt.ylim(0, max(df_agg['HIV_Prevalence']) * 1.3 if len(df_agg) else 100)

    for patch in ax.patches:
        ax.annotate(
            f"{patch.get_height():.2f}%",
            (patch.get_x() + patch.get_width() / 2.0, patch.get_height()),
            ha='center',
            va='bottom',
            xytext=(0, 5),
            textcoords='offset points',
            fontweight='bold',
            fontsize=12,
            color='#2c3e50',
        )

    sns.despine(left=True, bottom=False)
    plt.tight_layout()
    plt.show()

    total_without_tb = len(df_patients[df_patients['Has_TB'] == 0])
    hiv_without_tb = df_patients[df_patients['Has_TB'] == 0]['Has_HIV'].sum()
    total_with_tb = len(df_patients[df_patients['Has_TB'] == 1])
    hiv_with_tb = df_patients[df_patients['Has_TB'] == 1]['Has_HIV'].sum()

    print('=' * 80)
    print('SUMMARY OF OBSERVED DATA (TELESAP)')
    print('=' * 80)
    print(f" -> PATIENTS WITHOUT TUBERCULOSIS: {total_without_tb} (of these, {hiv_without_tb} have HIV)")
    print(f" -> PATIENTS WITH TUBERCULOSIS: {total_with_tb} (of these, {hiv_with_tb} have HIV)")
    print('=' * 80 + '\n')


# ---------------------------------------------------------------
# FIGURE 2 — Clinical journey of unique HIV+ patients
# ---------------------------------------------------------------
def figure_2_hiv_patient_journey(df: pd.DataFrame) -> None:
    col_tb = [c for c in df.columns if 'choice=Tuberculose' in c and 'infecciosas' in c.lower()][0]
    col_hiv = [c for c in df.columns if 'choice=HIV/AIDS' in c and 'infecciosas' in c.lower()][0]
    specialty_cols = [c for c in df.columns if 'Para qual especialidade' in c and 'choice=' in c]

    hiv_positive_ids = df[df[col_hiv] == 'Checked']['Record ID'].unique()
    plot_rows = []

    for patient_id in hiv_positive_ids:
        patient_history = df[df['Record ID'] == patient_id]
        had_tb = 'TB Positive' if (patient_history[col_tb] == 'Checked').any() else 'TB Negative'
        risk_color = 1 if had_tb == 'TB Positive' else 0

        patient_specialties = set()
        for specialty_col in specialty_cols:
            if (patient_history[specialty_col] == 'Checked').any():
                clean_name = specialty_col.split('choice=')[1].replace(')', '').strip()
                patient_specialties.add(clean_name)

        if len(patient_specialties) == 0:
            final_destination = 'No Referral (Resolved in Primary Care)'
        elif len(patient_specialties) == 1:
            final_destination = list(patient_specialties)[0]
        else:
            final_destination = 'Multiple Referrals (Historical Record)'

        plot_rows.append(
            {
                'Record ID': patient_id,
                'HIV_Status': 'HIV Positive',
                'TB_Status': had_tb,
                'Risk_Color': risk_color,
                'Specialty': final_destination,
            }
        )

    df_plot = pd.DataFrame(plot_rows)
    if df_plot.empty:
        print('No HIV-positive patients were found to generate the chart.')
        return

    top_specialties = df_plot['Specialty'].value_counts().nlargest(10).index
    df_plot['Specialty_Plot'] = np.where(
        df_plot['Specialty'].isin(top_specialties),
        df_plot['Specialty'],
        'Other Specialties',
    )

    fig = px.parallel_categories(
        df_plot,
        dimensions=['HIV_Status', 'TB_Status', 'Specialty_Plot'],
        color='Risk_Color',
        color_continuous_scale=[(0.0, '#3498db'), (1.0, '#e74c3c')],
        labels={
            'HIV_Status': 'Base Cohort (HIV+ Patients)',
            'TB_Status': 'TB Diagnosis',
            'Specialty_Plot': 'Patient Clinical Destination',
        },
        title=f"Clinical Journey of HIV+ Patients (Total: {len(df_plot)} Unique Patients)",
    )

    fig.update_layout(
        font=dict(size=13, family='Arial'),
        coloraxis_showscale=False,
        margin=dict(l=50, r=50, t=80, b=50),
    )
    fig.show()

    df_table = (
        df_plot.groupby(['HIV_Status', 'TB_Status', 'Specialty_Plot'])
        .size()
        .reset_index(name='Patient_Count')
        .sort_values(by=['TB_Status', 'Patient_Count'], ascending=[False, False])
        .reset_index(drop=True)
    )

    print('\n' + '=' * 85)
    print(f"FLOW TABLE: HIV+ PATIENT CLINICAL JOURNEY (N={len(df_plot)} Unique Patients)")
    print('=' * 85)
    print(df_table.to_string(index=False))
    print('=' * 85)

    output_name = 'HIV_Patient_Journey_Flow_Table_Unique_Patients_EN.csv'
    df_table.to_csv(output_name, index=False, encoding='utf-8-sig')
    print(f"\n[SUCCESS] The full chart table was saved as '{output_name}'.")


# ---------------------------------------------------------------
# FIGURE 3 — Age distribution by CIAP in the HIV+ population
# ---------------------------------------------------------------
def figure_3_age_by_ciap_hiv(df: pd.DataFrame) -> None:
    age_col = 'Idade'
    hiv_col = [c for c in df.columns if 'choice=HIV/AIDS' in c and 'infecciosas' in c.lower()][0]

    ciap_cols = [c for c in df.columns if 'ciap' in c.lower()]
    if not ciap_cols:
        raise ValueError('No CIAP column was found.')
    ciap_col = ciap_cols[0]

    df = df.copy()
    df['Age_Num'] = pd.to_numeric(df[age_col], errors='coerce')
    df = df.sort_values(['Record ID', 'Repeat Instance'], na_position='first')
    df[['Age_Num', hiv_col]] = df.groupby('Record ID')[['Age_Num', hiv_col]].ffill()

    df_visits = df[df['Repeat Instrument'].notna()].copy()
    df_hiv = df_visits[df_visits[hiv_col] == 'Checked'].copy()
    df_hiv = df_hiv.dropna(subset=['Age_Num', ciap_col])
    df_hiv = df_hiv[df_hiv[ciap_col].astype(str).str.strip() != '']
    df_hiv = df_hiv[df_hiv[ciap_col].astype(str).str.lower() != 'nan']

    if df_hiv.empty:
        print('No valid visits were found for HIV patients (or CIAP/Age fields are missing).')
        return

    df_hiv['CIAP_Clean'] = (
        df_hiv[ciap_col].astype(str).str.strip().apply(lambda x: x[:40] + '...' if len(x) > 40 else x)
    )

    print('\n' + '=' * 80)
    print(f"DIAGNOSTIC PROFILE (HIV-POSITIVE POPULATION: {len(df_hiv)} valid visits)")
    print('=' * 80)
    ranking_ciap = df_hiv['CIAP_Clean'].value_counts()
    for ciap, count in ranking_ciap.head(10).items():
        pct = (count / len(df_hiv)) * 100
        print(f" - {str(ciap)[:50]:<50}: {count} occurrences ({pct:.1f}%)")
    print('=' * 80)

    top_8_ciaps = ranking_ciap.nlargest(8).index
    df_plot = df_hiv[df_hiv['CIAP_Clean'].isin(top_8_ciaps)].copy()

    fig = go.Figure()
    colors = px.colors.sequential.Reds[::-1]

    for i, ciap in enumerate(reversed(top_8_ciaps)):
        age_array = df_plot[df_plot['CIAP_Clean'] == ciap]['Age_Num']
        current_color = colors[(i + 2) % len(colors)]

        fig.add_trace(
            go.Violin(
                x=age_array,
                name=ciap,
                side='positive',
                width=1.5,
                line_color='black',
                line_width=1.2,
                fillcolor=current_color,
                opacity=0.85,
                points=False,
                meanline_visible=True,
                spanmode='hard',
            )
        )

    fig.update_layout(
        title=dict(
            text=f"Age Distribution of the Main CIAP Codes (HIV+ Population, N={len(df_hiv)} visits)",
            font=dict(size=18, family='Arial'),
        ),
        xaxis=dict(
            title='Patient Age (Years)',
            showgrid=True,
            gridcolor='#e0e0e0',
            zeroline=False,
            title_font=dict(size=14),
        ),
        yaxis=dict(showgrid=False, zeroline=False),
        height=700,
        margin=dict(l=300, r=50, t=80, b=80),
        showlegend=False,
        plot_bgcolor='white',
        paper_bgcolor='white',
    )
    fig.show()

    df_table = (
        df_plot.groupby('CIAP_Clean')['Age_Num']
        .agg(
            Visit_Count='count',
            Minimum_Age='min',
            Mean_Age='mean',
            Median_Age='median',
            Maximum_Age='max',
        )
        .reset_index()
    )
    df_table['Mean_Age'] = df_table['Mean_Age'].round(1)
    df_table = df_table.sort_values(by='Visit_Count', ascending=False).reset_index(drop=True)

    print('\n' + '=' * 95)
    print('STATISTICAL TABLE: AGE DISTRIBUTION BY CIAP (Top 8 - HIV+ Population)')
    print('=' * 95)
    print(df_table.to_string(index=False))
    print('=' * 95)

    output_name = 'Ridgeline_Age_by_CIAP_HIV_EN.csv'
    df_table.to_csv(output_name, index=False, encoding='utf-8-sig')
    print(f"\n[SUCCESS] The chart statistical table was saved as '{output_name}'.")


# ---------------------------------------------------------------
# RUN EVERYTHING
# ---------------------------------------------------------------
def main():
    df = load_csv()
    figure_1_hiv_tb_coinfection(df)
    figure_2_hiv_patient_journey(df)
    figure_3_age_by_ciap_hiv(df)

main()